
# Synthetic 100-sample binary text classification with a scratch Transformer and D-NGD, SGD and AdamW

This notebook creates a **100-sample artificial text dataset**, trains a **tiny Transformer from scratch**, imports the **DNGD, SGD and AdamW** optimizer implementation from the existing project source file, and plots:

- training loss vs iteration
- test loss vs iteration
- training classification error vs iteration
- test classification error vs iteration

Notes:
- The dataset is intentionally tiny: **80 training samples + 20 test samples**.
- Optimizers are loaded **from the project's `optimizers/optimizers.py` source file**.

Run:
- Select the corresponding optimizer in the variable **optimizer_name** from the following code block, then run all.


In [ ]:
# optimizer_name = "SGD"
# optimizer_name = "DNGD"
optimizer_name = "AdamW"

In [ ]:

import sys
import math
import random
import types
import importlib.util
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

# Try to locate the project root.
CANDIDATES = [Path.cwd(), Path.cwd() / 'code_inspect', Path('/mnt/data/code_inspect')]
PROJECT_ROOT = None
for cand in CANDIDATES:
    if (cand / 'optimizers').exists() and (cand / 'utils').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot find the project root containing optimizers/ and utils/.')
print('project root:', PROJECT_ROOT)


In [ ]:

# Load DNGD directly from the project's source file.
# This uses the existing DNGD implementation while bypassing the heavy package-level imports.
utils_pkg = types.ModuleType('utils')
utils_pkg.__path__ = [str(PROJECT_ROOT / 'utils')]
sys.modules['utils'] = utils_pkg

kfac_spec = importlib.util.spec_from_file_location('utils.kfac_utils', PROJECT_ROOT / 'utils' / 'kfac_utils.py')
kfac_module = importlib.util.module_from_spec(kfac_spec)
sys.modules['utils.kfac_utils'] = kfac_module
kfac_spec.loader.exec_module(kfac_module)

optim_spec = importlib.util.spec_from_file_location('project_optimizers_module', PROJECT_ROOT / 'optimizers' / 'optimizers.py')
optim_module = importlib.util.module_from_spec(optim_spec)
sys.modules['project_optimizers_module'] = optim_module
optim_spec.loader.exec_module(optim_module)

if optimizer_name == "SGD":
    SGD = optim_module.SGD_
    print('Loaded SGD from:', PROJECT_ROOT / 'optimizers' / 'optimizers.py')
elif optimizer_name == "DNGD":
    DNGD = optim_module.DNGD
    print('Loaded DNGD from:', PROJECT_ROOT / 'optimizers' / 'optimizers.py')
elif optimizer_name == "AdamW":
    AdamW = optim_module.AdamW
    print('Loaded AdamW from:', PROJECT_ROOT / 'optimizers' / 'optimizers.py')


In [ ]:

# -------------------------------------------------
# 1. Build a tiny synthetic 2-class text dataset.
# -------------------------------------------------
# The label is determined mainly by the FIRST token:
#   'a x' and 'a y' -> class 1
#   'b x' and 'b y' -> class 0
#
# This is intentionally simple, so the scratch Transformer + DNGD experiment
# runs quickly and gives clean curves.

base_patterns = [
    ('a x', 1),
    ('b x', 0),
    ('a y', 1),
    ('b y', 0),
]

samples = []
for _ in range(25):
    samples.extend(base_patterns)
assert len(samples) == 100
random.shuffle(samples)

print('Number of samples:', len(samples))
print('First 10 samples:', samples[:10])


In [ ]:

# Tokenize into a tiny vocabulary.
vocab = {
    '<pad>': 0,
    'a': 1,
    'b': 2,
    'x': 3,
    'y': 4,
}
MAX_LEN = 2
PAD_ID = vocab['<pad>']


def encode(text):
    ids = [vocab[token] for token in text.split()]
    mask = [1] * len(ids)
    while len(ids) < MAX_LEN:
        ids.append(PAD_ID)
        mask.append(0)
    return ids[:MAX_LEN], mask[:MAX_LEN]

# Stratified split: 80 train / 20 test
positive = [x for x in samples if x[1] == 1]
negative = [x for x in samples if x[1] == 0]
train_samples = positive[:40] + negative[:40]
test_samples = positive[40:] + negative[40:]
random.shuffle(train_samples)
random.shuffle(test_samples)

print('Train size:', len(train_samples), 'Test size:', len(test_samples))


def tensorize(data):
    input_ids = []
    attention_mask = []
    labels = []
    for text, label in data:
        ids, mask = encode(text)
        input_ids.append(ids)
        attention_mask.append(mask)
        labels.append(label)
    return (
        torch.tensor(input_ids, dtype=torch.long, device=DEVICE),
        torch.tensor(attention_mask, dtype=torch.long, device=DEVICE),
        torch.tensor(labels, dtype=torch.long, device=DEVICE),
    )

train_input_ids, train_attention_mask, train_labels = tensorize(train_samples)
test_input_ids, test_attention_mask, test_labels = tensorize(test_samples)


In [ ]:

# -------------------------------------------------
# 2. Define a scratch Transformer classifier.
# -------------------------------------------------
# Important: q/k/v/o are explicit nn.Linear modules so DNGD can act on them.

class TinyTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, max_len, d_model=4, num_heads=2, num_classes=2):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, max_len, d_model))

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.o_proj = nn.Linear(d_model, d_model)

        self.classifier = nn.Linear(d_model, num_classes)
        self.reset_parameters()

    def reset_parameters(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.pos_embedding, mean=0.0, std=0.02)

    def _split_heads(self, x):
        bsz, seq_len, _ = x.shape
        x = x.view(bsz, seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def forward(self, input_ids, attention_mask):
        x = self.token_embedding(input_ids) + self.pos_embedding[:, :input_ids.size(1), :]

        q = self._split_heads(self.q_proj(x))
        k = self._split_heads(self.k_proj(x))
        v = self._split_heads(self.v_proj(x))

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        probs = torch.softmax(scores, dim=-1)
        context = torch.matmul(probs, v)
        context = context.transpose(1, 2).contiguous().view(x.size(0), x.size(1), self.d_model)

        x = x + self.o_proj(context)

        # Use the first token representation as the classification token.
        pooled = x[:, 0, :]
        logits = self.classifier(pooled)
        return logits


In [ ]:

# -------------------------------------------------
# 3. Train with DNGD and record curves.
# -------------------------------------------------

model = TinyTransformerClassifier(vocab_size=len(vocab), max_len=MAX_LEN).to(DEVICE)
criterion = nn.CrossEntropyLoss()
if optimizer_name == "SGD":
    optimizer = SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
elif optimizer_name == "DNGD":
    optimizer = DNGD(model, lr=0.1, momentum=0.6, weight_decay=5e-4, damping=0.5)
elif optimizer_name == "AdamW":
    optimizer = AdamW(model.parameters(), lr=0.1,weight_decay=5e-4)
NUM_ITERATIONS = 12
history = {
    'iteration': [],
    'train_loss': [],
    'test_loss': [],
    'train_error': [],
    'test_error': [],
}

@torch.no_grad()
def evaluate_split(input_ids, attention_mask, labels):
    model.eval()
    logits = model(input_ids=input_ids, attention_mask=attention_mask)
    loss = criterion(logits, labels).item()
    preds = torch.argmax(logits, dim=-1)
    error = 1.0 - (preds == labels).float().mean().item()
    return loss, error

for iteration in range(NUM_ITERATIONS + 1):
    if iteration > 0:
        model.train()
        optimizer.zero_grad()
        logits = model(input_ids=train_input_ids, attention_mask=train_attention_mask)
        loss = criterion(logits, train_labels)
        loss.backward()
        optimizer.step()

    train_loss, train_error = evaluate_split(train_input_ids, train_attention_mask, train_labels)
    test_loss, test_error = evaluate_split(test_input_ids, test_attention_mask, test_labels)

    history['iteration'].append(iteration)
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['train_error'].append(train_error)
    history['test_error'].append(test_error)

    print(
        f"iter={iteration:02d} | "
        f"train_loss={train_loss:.4f} | test_loss={test_loss:.4f} | "
        f"train_error={train_error:.4f} | test_error={test_error:.4f}"
    )


In [ ]:

# -------------------------------------------------
# 4. Plot the requested figures.
# -------------------------------------------------

iterations = history['iteration']
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(2, 2, 1)
ax1.plot(iterations, history['train_loss'])
ax1.set_title('Training Loss')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Loss')

ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(iterations, history['test_loss'])
ax2.set_title('Test Loss')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Loss')

ax3 = fig.add_subplot(2, 2, 3)
ax3.plot(iterations, history['train_error'])
ax3.set_title('Training Classification Error')
ax3.set_xlabel('Iteration')
ax3.set_ylabel('Error Rate')

ax4 = fig.add_subplot(2, 2, 4)
ax4.plot(iterations, history['test_error'])
ax4.set_title('Test Classification Error')
ax4.set_xlabel('Iteration')
ax4.set_ylabel('Error Rate')

fig.tight_layout()
plt.show()


In [ ]:

# Optional: save the figure.
figure_path = Path('scratch_transformer_curves.png')
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
print('Saved figure to:', figure_path.resolve())
